## Imports

In [ ]:
import json
import time
import uuid
from datetime import datetime, timezone

import pandas as pd
from kafka import KafkaProducer

## Load Dataset

In [ ]:
df = pd.read_csv("../data/raw/creditcard.csv")

print(df.shape)

df.head()

## Create Kafka Producer

In [ ]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

print("Kafka Producer Connected")

## Stream Transactions

In [ ]:
topic = "fraud-transactions"

for index, row in df.head(100).iterrows():

    transaction = row.to_dict()

    # Production metadata
    transaction["transaction_id"] = str(uuid.uuid4())

    transaction["event_timestamp"] = (
        datetime.now(timezone.utc)
        .isoformat(timespec="milliseconds")
    )

    producer.send(
        topic,
        value=transaction
    )

    print(
        f"Sent Transaction {index + 1} | "
        f"ID: {transaction['transaction_id']}"
    )

    time.sleep(1)

## Flush Messages

In [ ]:
producer.flush()
producer.close()

print("Streaming Finished")